# Raw Dataset Processor

Pipeline monitoring untuk mengubah raw dataset `datasets/url_discovery/*.json` menjadi kandidat dataset dengan schema seperti `datasets/v1_shs_datasets.csv`. Notebook ini tidak menyimpan output secara otomatis.

## 1. Setup

In [22]:
from pathlib import Path
import sys

import polars as pl
from IPython.display import display


def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in (current, *current.parents):
        if (candidate / "config.py").exists() and (candidate / "services").is_dir():
            return candidate
    raise FileNotFoundError("Root proyek tidak ditemukan")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import config
from services.dataset_service import DatasetService

pl.Config.set_tbl_hide_column_data_types(True)
pl.Config.set_tbl_cell_alignment("LEFT")
pl.Config.set_fmt_str_lengths(250)
pl.Config.set_tbl_width_chars(180)
pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_rows(20)

dataset_service = DatasetService()
RAW_FOLDER = config.DATASETS / "url_discovery"
RESEARCH_CONFIG_PATH = PROJECT_ROOT / "research_config.json"
REFERENCE_DATASET_PATH = config.DATASETS / "v1_shs_datasets.csv"

print(f"Raw folder: {RAW_FOLDER}")
print(f"Research config: {RESEARCH_CONFIG_PATH}")
print(f"Reference schema: {REFERENCE_DATASET_PATH}")

Raw folder: E:\School\tugas-akhir\project\datasets\url_discovery
Research config: E:\School\tugas-akhir\project\research_config.json
Reference schema: E:\School\tugas-akhir\project\datasets\v1_shs_datasets.csv


## 2. Load Config dan Raw Discovery

In [23]:
research_config = dataset_service.load_research_config(RESEARCH_CONFIG_PATH)
meta_df = dataset_service.load_url_discovery_meta(RAW_FOLDER)
queries_df = dataset_service.load_url_discovery_queries(RAW_FOLDER)
raw_records_df = dataset_service.load_url_discovery_records(RAW_FOLDER)

RECORDS_FILTER_CONDITIONS = [
    # pl.col("content_status") == "success",
    # pl.col("content_text").fill_null("").str.strip_chars().str.len_chars() > 0,
    # ~pl.col("domain").str.contains("instagram.com|tiktok.com|facebook.com"),
]


def apply_filter_conditions(df: pl.DataFrame, conditions: list) -> pl.DataFrame:
    if not conditions:
        return df
    expression = conditions[0]
    for condition in conditions[1:]:
        expression = expression & condition
    return df.filter(expression)


records_df = apply_filter_conditions(raw_records_df, RECORDS_FILTER_CONDITIONS)

print("Search label:", research_config.get("search_label"))
print("Primary location:", research_config.get("primary_location"))
print(f"File metadata: {meta_df.height:,}")
print(f"Query rows: {queries_df.height:,}")
print(f"Raw records unik: {raw_records_df.height:,}")
print(f"Records setelah filter manual: {records_df.height:,}")

display(meta_df)

Search label: SHS/PLTS Kalimantan Barat
Primary location: Kalimantan Barat
File metadata: 6
Query rows: 1,310
Raw records unik: 855
Records setelah filter manual: 855


source_file,built_at,search_label,source_files,record_count,content_count,query_count
"""dataset_20260627T094855Z.json""","""2026-06-27T09:48:55.240893+00:00""","""SHS/PLTS Kalimantan Barat""","{""E:\Work\personal\url-discovery\data\urls.json"",""E:\Work\personal\url-discovery\data\contents.json"",""E:\Work\personal\url-discovery\data\search_state.json""}",214,214,335
"""dataset_20260627T101253Z.json""","""2026-06-27T10:12:53.267116+00:00""","""SHS/PLTS Kalimantan Barat""","{""E:\Work\personal\url-discovery\data\urls.json"",""E:\Work\personal\url-discovery\data\contents.json"",""E:\Work\personal\url-discovery\data\search_state.json""}",203,203,268
"""dataset_20260627T135259Z.json""","""2026-06-27T13:52:59.725144+00:00""","""SHS/PLTS Kalimantan Barat""","{""E:\Work\personal\url-discovery\data\urls.json"",""E:\Work\personal\url-discovery\data\contents.json"",""E:\Work\personal\url-discovery\data\search_state.json""}",88,88,197
"""dataset_20260628T033631Z.json""","""2026-06-28T03:36:31.219965+00:00""","""SHS/PLTS Kalimantan Barat""","{""E:\Work\personal\url-discovery\data\urls.json"",""E:\Work\personal\url-discovery\data\contents.json"",""E:\Work\personal\url-discovery\data\search_state.json""}",202,202,381
"""dataset_20260628T104512Z.json""","""2026-06-28T10:45:12.810124+00:00""","""SHS/PLTS Kalimantan Barat""","{""E:\Work\personal\url-discovery\data\urls.json"",""E:\Work\personal\url-discovery\data\contents.json"",""E:\Work\personal\url-discovery\data\search_state.json""}",136,136,369
"""dataset_20260628T113525Z.json""","""2026-06-28T11:35:25.172805+00:00""","""SHS/PLTS Kalimantan Barat""","{""E:\Work\personal\url-discovery\data\urls.json"",""E:\Work\personal\url-discovery\data\contents.json"",""E:\Work\personal\url-discovery\data\search_state.json""}",12,12,28


## 3. Monitoring Kualitas Raw Dataset

In [24]:
status_distribution = (
    records_df.with_columns(pl.col("content_status").fill_null("missing"))
    .group_by("content_status")
    .agg(pl.len().alias("jumlah"))
    .sort("jumlah", descending=True)
)

domain_distribution = (
    records_df.with_columns(pl.col("domain").fill_null("unknown"))
    .group_by("domain")
    .agg(pl.len().alias("jumlah"))
    .sort("jumlah", descending=True)
    .head(20)
)

query_group_distribution = (
    records_df.with_columns(pl.col("query_group").fill_null("unknown"))
    .group_by("query_group")
    .agg(pl.len().alias("jumlah"))
    .sort("jumlah", descending=True)
    .head(20)
)

text_length_summary = records_df.select(
    pl.len().alias("total_records"),
    (
        pl.col("content_text")
        .fill_null("")
        .str.strip_chars()
        .str.len_chars()
        > 0
    ).sum().alias("records_with_text"),
    pl.col("content_text_length").min().alias("min_text_length"),
    pl.col("content_text_length").mean().round(2).alias("avg_text_length"),
    pl.col("content_text_length").max().alias("max_text_length"),
)

display(status_distribution)
display(domain_distribution)
display(query_group_distribution)
display(text_length_summary)

content_status,jumlah
"""success""",645
"""too_short""",124
"""pdf_skipped""",45
"""failed""",41


domain,jumlah
"""www.instagram.com""",340
"""www.tiktok.com""",123
"""www.facebook.com""",25
"""rri.co.id""",21
"""kalbar.antaranews.com""",20
"""pontianakpost.jawapos.com""",19
"""www.researchgate.net""",9
"""hijau.bisnis.com""",8
"""gatrik.esdm.go.id""",8
"""id.scribd.com""",8


query_group,jumlah
"""issue:benefit""",93
"""issue:problem""",84
"""social""",75
"""issue:access""",62
"""issue:environment""",55
"""issue:experience""",50
"""entity-opinion""",48
"""issue:socioeconomic""",47
"""issue:funding""",46
"""pdf""",39


total_records,records_with_text,min_text_length,avg_text_length,max_text_length
855,665,0,3076.92,33760


## 4. Bangun Kandidat Schema v1

In [25]:
candidate_df = dataset_service.build_v1_candidate_rows(
    records_df=records_df,
    research_config=research_config,
)

CANDIDATE_FILTER_CONDITIONS = [
    # pl.col("location") == "",
    # pl.col("source_url").str.to_lowercase().str.contains("facebook.com"),
    # ~pl.col("text").str.to_lowercase().str.contains("penayangan"),
    # pl.col("aspect").is_in(["experience", "benefit", "access"]),
]

filtered_candidate_df = apply_filter_conditions(candidate_df, CANDIDATE_FILTER_CONDITIONS)

reference_df = dataset_service.load(REFERENCE_DATASET_PATH)
expected_columns = list(dataset_service.v1_dataset_columns())
missing_candidate_columns = [column for column in expected_columns if column not in filtered_candidate_df.columns]
extra_candidate_columns = [column for column in filtered_candidate_df.columns if column not in expected_columns]

print(f"Candidate rows sebelum filter kandidat: {candidate_df.height:,}")
print(f"Candidate rows setelah filter kandidat: {filtered_candidate_df.height:,}")
print("Kolom wajib hilang:", missing_candidate_columns)
print("Kolom monitoring tambahan:", extra_candidate_columns)
print("Urutan kolom utama sama:", filtered_candidate_df.columns[: len(expected_columns)] == expected_columns)

display(filtered_candidate_df.select(expected_columns).head(20))

Candidate rows sebelum filter kandidat: 855
Candidate rows setelah filter kandidat: 855
Kolom wajib hilang: []
Kolom monitoring tambahan: ['raw_source_file', 'raw_domain', 'content_status', 'query_group', 'query', 'raw_title', 'raw_text_length']
Urutan kolom utama sama: True


text_id,text,subjectivity_type,speaker_type,public_opinion_scope,corpus_role,aspect,location,sentiment_label,label_status,source_id,source_type,source_url,dataset_tier,inclusion_status,verification_status,evidence_support_score,parent_text_id,decision_note
"""RAW-0001""","""Program Listrik Desa di Papua Hadirkan Terang dan Harapan Hingga Pelosok Negeri Program Listrik Desa di Papua Hadirkan Terang dan Harapan ... suarapemredkalbar.com https://www.suarapemredkalbar.com › read › nasional 26062026 program listrik desa di p…","""public_expectation""","""community_representative""","""public_opinion""","""core_public_opinion""","""experience""","""Singkawang""","""""","""unlabeled""","""RAW-SRC-0001""","""online_news""","""https://www.suarapemredkalbar.com/read/nasional/26062026/program-listrik-desa-di-papua-hadirkan-terang-dan-harapan-hingga-pelosok-negeri""","""A_candidate_core""","""candidate_analysis_ready""","""perlu_verifikasi""",1.0,"""""","""candidate_from_raw_discovery"""
"""RAW-0002""","""Ari Yunianto on Instagram: ""Bekasi beres. Semangat…!! Antrian project installasi PLTS masih full sampai Agustus #plts #panelsurya #offgrid #pltsatap #solarpanels"" Bekasi beres. Semangat…!! Antrian project installasi PLTS ... Instagram · ariyuniant0 2…","""public_experience""","""community_representative""","""public_opinion""","""core_public_opinion""","""general_shs""","""Sambas""","""""","""unlabeled""","""RAW-SRC-0002""","""online_news""","""https://www.instagram.com/reel/DaCiaOBxC_d""","""A_candidate_core""","""candidate_analysis_ready""","""perlu_verifikasi""",1.0,"""""","""candidate_from_raw_discovery"""
"""RAW-0003""","""OIS POWER on Instagram: ""#PLTS #PanelSurya #SolarPanel #EnergiTerbarukan #SolarEnergy"" OIS POWER | #PLTS #PanelSurya #SolarPanel ... Instagram · ois_power 6 suka · 1 bulan yang lalu 6 likes, 0 comments - ois_power on May 26, 2026: ""#PLTS #PanelSurya …","""public_experience""","""community_representative""","""public_opinion""","""core_public_opinion""","""general_shs""","""Sambas""","""""","""unlabeled""","""RAW-SRC-0003""","""online_news""","""https://www.instagram.com/reel/DY0bixATBPD""","""A_candidate_core""","""candidate_analysis_ready""","""perlu_verifikasi""",1.0,"""""","""candidate_from_raw_discovery"""
"""RAW-0004""","""TEKNIK LISTRIK on Instagram: ""Panel Surya Listrik (Fotovoltaik/PV) adalah teknologi yang mengambil energi dari cahaya matahari (foton), bukan panas, lalu mengubahnya menjadi listrik DC yang kemudian diubah oleh Inverter menjadi listrik AC yang dapat …","""public_experience""","""community_representative""","""public_opinion""","""core_public_opinion""","""general_shs""","""Sambas""","""""","""unlabeled""","""RAW-SRC-0004""","""online_news""","""https://www.instagram.com/p/DRq794LgAB2""","""A_candidate_core""","""candidate_analysis_ready""","""perlu_verifikasi""",1.0,"""""","""candidate_from_raw_discovery"""
"""RAW-0005""","""Datascrip Service Center on Instagram: ""Waktu yg tepat untuk membersihkan panel surya."" Waktu yg tepat untuk membersihkan panel surya. Instagram · datascrip.service.center.id 1 suka · 7 bulan yang lalu 1 likes, 0 comments - datascrip.service.center.i…","""public_experience""","""community_representative""","""public_opinion""","""core_public_opinion""","""general_shs""","""Sambas""","""""","""unlabeled""","""RAW-SRC-0005""","""online_news""","""https://www.instagram.com/reel/DQlfNxOATsq""","""A_candidate_core""","""candidate_analysis_ready""","""perlu_verifikasi""",1.0,"""""","""candidate_from_raw_discovery"""
"""RAW-0006""","""TikTok - Make Your Day Listrik Padam Sumatera: Solusi PLTS dan Stok Cadangan ... TikTok · evanstoryy 7,2 rb+ penayangan @evanstoryyy 7643019382251080967""","""contextual_source""","""community_representative""","""contextual_reference""","""excluded""","""general_shs""","""Sambas""","""""","""unlabeled""","""RAW-SRC-0006""","""online_news""","""https://www.tiktok.com/@evanstoryyy/video/7643019382251080967""","""C_holdout_excluded""","""he

## 5. Monitoring Kandidat

In [26]:
tier_distribution = (
    filtered_candidate_df.group_by("dataset_tier")
    .agg(pl.len().alias("jumlah"))
    .sort("jumlah", descending=True)
)

inclusion_distribution = (
    filtered_candidate_df.group_by("inclusion_status")
    .agg(pl.len().alias("jumlah"))
    .sort("jumlah", descending=True)
)

source_type_distribution = (
    filtered_candidate_df.group_by("source_type")
    .agg(pl.len().alias("jumlah"))
    .sort("jumlah", descending=True)
)

aspect_distribution = (
    filtered_candidate_df.group_by("aspect")
    .agg(pl.len().alias("jumlah"))
    .sort("jumlah", descending=True)
    .head(20)
)


location_distribution = (
    filtered_candidate_df.group_by("location")
    .agg(pl.len().alias("jumlah"))
    .sort("jumlah", descending=True)
    .head(20)
)

display(tier_distribution)
display(inclusion_distribution)
display(source_type_distribution)
display(aspect_distribution)
display(location_distribution)

dataset_tier,jumlah
"""A_candidate_core""",645
"""C_holdout_excluded""",210


inclusion_status,jumlah
"""candidate_analysis_ready""",645
"""held_out_not_for_sentiment_core""",210


source_type,jumlah
"""online_news""",758
"""academic_repository""",57
"""government_official""",40


aspect,jumlah
"""experience""",388
"""general_shs""",266
"""benefit""",68
"""environment""",43
"""grid_hybrid""",34
"""electricity_access""",29
"""problem""",9
"""household_solar""",7
"""access""",7
"""village_solar""",3


location,jumlah
"""Kalimantan Barat""",168
"""""",143
"""Pontianak""",66
"""Kayong Utara""",56
"""Bengkayang""",50
"""Kapuas Hulu""",50
"""Kubu Raya""",49
"""Ketapang""",48
"""Sintang""",40
"""Sambas""",39


## 6. Review Kandidat Prioritas

In [27]:
review_columns = [
    "text_id",
    "dataset_tier",
    "inclusion_status",
    "evidence_support_score",
    "aspect",
    "location",
    "source_type",
    "source_url",
    "text",
    "decision_note",
    "query_group",
    "query",
]

priority_df = (
    filtered_candidate_df
    .filter(pl.col("dataset_tier").is_in(["A_candidate_core", "B_review_queue"]))
    .sort(["dataset_tier", "evidence_support_score"], descending=[False, True])
    .select(review_columns)
)

display(priority_df.head(50))

text_id,dataset_tier,inclusion_status,evidence_support_score,aspect,location,source_type,source_url,text,decision_note,query_group,query
"""RAW-0440""","""A_candidate_core""","""candidate_analysis_ready""",1.0,"""experience""","""Kapuas Hulu""","""online_news""","""https://pena.co.id/2025/08/28/gangguan-listrik-di-putussibau-ribuan-pelanggan-terdampak""","""Gangguan Listrik di Putussibau, Ribuan Pelanggan Terdampak - pena.co.id Gangguan Listrik di Putussibau, Ribuan Pelanggan ... PENA.co.id https://pena.co.id › 2025/08/28 › gangguan-listrik-di-p... KAPUAS HULU, PENA.co.id – Listrik di sejumlah wilayah d…","""candidate_from_raw_discovery""","""issue:experience""","""""listrik desa"" ""tanggapan masyarakat"" ""Putussibau"" after:2024-01-01 before:2026-06-27"""
"""RAW-0360""","""A_candidate_core""","""candidate_analysis_ready""",1.0,"""general_shs""","""Pontianak""","""online_news""","""https://www.instagram.com/reel/DZzCZNnSIt1""","""Post isn't available • Instagram Ketika banyak orang khawatir listrik padam, rumah ini sudah ... Instagram · warungenergi 1 suka · 6 hari yang lalu reel DZzCZNnSIt1""","""candidate_from_raw_discovery""","""issue:benefit""","""""Solar Home System"" ""UMKM terbantu surya"" ""Pontianak"" after:2024-01-01 before:2026-06-27"""
"""RAW-0003""","""A_candidate_core""","""candidate_analysis_ready""",1.0,"""general_shs""","""Sambas""","""online_news""","""https://www.instagram.com/reel/DY0bixATBPD""","""OIS POWER on Instagram: ""#PLTS #PanelSurya #SolarPanel #EnergiTerbarukan #SolarEnergy"" OIS POWER | #PLTS #PanelSurya #SolarPanel ... Instagram · ois_power 6 suka · 1 bulan yang lalu 6 likes, 0 comments - ois_power on May 26, 2026: ""#PLTS #PanelSurya …","""candidate_from_raw_discovery""","""issue:benefit""","""""PLTS"" ""hemat tagihan listrik"" ""Sambas"" after:2024-01-01 before:2026-06-27"""
"""RAW-0004""","""A_candidate_core""","""candidate_analysis_ready""",1.0,"""general_shs""","""Sambas""","""online_news""","""https://www.instagram.com/p/DRq794LgAB2""","""TEKNIK LISTRIK on Instagram: ""Panel Surya Listrik (Fotovoltaik/PV) adalah teknologi yang mengambil energi dari cahaya matahari (foton), bukan panas, lalu mengubahnya menjadi listrik DC yang kemudian diubah oleh Inverter menjadi listrik AC yang dapat …","""candidate_from_raw_discovery""","""issue:benefit""","""""PLTS"" ""hemat tagihan listrik"" ""Sambas"" after:2024-01-01 before:2026-06-27"""
"""RAW-0005""","""A_candidate_core""","""candidate_analysis_ready""",1.0,"""general_shs""","""Sambas""","""online_news""","""https://www.instagram.com/reel/DQlfNxOATsq""","""Datascrip Service Center on Instagram: ""Waktu yg tepat untuk membersihkan panel surya."" Waktu yg tepat untuk membersihkan panel surya. Instagram · datascrip.service.center.id 1 suka · 7 bulan yang lalu 1 likes, 0 comments - datascrip.service.center.i…","""candidate_from_raw_discovery""","""issue:benefit""","""""PLTS"" ""hemat tagihan listrik"" ""Sambas"" after:2024-01-01 before:2026-06-27"""
"""RAW-0007""","""A_candidate_core""","""candidate_analysis_ready""",1.0,"""general_shs""","""Sambas""","""online_news""","""https://www.instagram.com/p/DP2n1gQD6J_""","""Damai Cable Indonesia on Instagram: ""Butuh kabel panel surya? Kami siap penuhi kebutuhan proyek energi terbarukanmu! Kuat, awet, dan sudah teruji kualitasnya. #damaicable"" Butuh kabel panel surya? Kami siap penuhi kebutuhan ... Instagram · damaicable…","""candidate_from_raw_discovery""","""issue:benefit""","""""tenaga surya"" ""listrik 24 jam"" ""Sambas"" after:2024-01-01 before:2026-06-27"""
"""RAW-0445""","""A_candidate_core""","""candidate_analysis_ready""",1.0,"""problem""","""Bengkayang""","""online_news""","""https://hartechsby.co.id/bagaimana-generator-menghasilkan-listrik-penjelasan-dan-cara-kerjanya""","""Bagaimana Generator Menghasilkan Listrik? Ini Penjelasannya Bagaimana Generator Menghasilkan Listrik? Ini ... Hartech SBY https://hartechsby.co.id › Blog Temukan cara generator menghasilkan listrik dalam artikel ini. Pelajari

## 7. Export Manual Opsional

In [28]:
EXPORT_CSV = True
EXPORT_PATH = config.OUTPUTS / "datasets" / "raw_candidate_schema.csv"

if EXPORT_CSV:
    EXPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
    filtered_candidate_df.write_csv(EXPORT_PATH)
    print(f"Candidate dataset terfilter disimpan ke: {EXPORT_PATH}")
else:
    print("Export dimatikan. Ubah EXPORT_CSV = True jika ingin menyimpan kandidat terfilter secara manual.")

Candidate dataset terfilter disimpan ke: E:\School\tugas-akhir\project\outputs\datasets\raw_candidate_schema.csv
